In [1]:
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# ==============================================================================
# 1. CONFIGURACIÓN Y CONSTANTES
# ==============================================================================

# Los 12 slugs exactos estipulados en TAXONOMIA.md (minúsculas y sin acentos)
SLUGS_M1 = [
    "alimentacion",
    "transporte",
    "vivienda",
    "servicios",
    "salud",
    "educacion",
    "entretenimiento",
    "compras",
    "finanzas",
    "ahorro_inversion",
    "ingresos",
    "otros"
]

# Las 3 etiquetas exactas para el Perfil de Salud Financiera (M2)
ETIQUETAS_M2 = ["saludable", "en_observacion", "en_riesgo"]

RANDOM_STATE = 42

# ==============================================================================
# 2. MODELO M1: CLASIFICADOR DE CATEGORÍAS POR DESCRIPCIÓN
# ==============================================================================

def construir_pipeline_m1():
    """
    Construye el pipeline TF-IDF (word + char_wb) + LogisticRegression
    que acepta únicamente la cadena de texto de la descripción.
    """
    word_vectorizer = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        lowercase=True,
        strip_accents='unicode'
    )

    char_vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        lowercase=True,
        strip_accents='unicode'
    )

    union_features = FeatureUnion([
        ('word_tfidf', word_vectorizer),
        ('char_tfidf', char_vectorizer)
    ])

    pipeline = Pipeline([
        ('features', union_features),
        ('classifier', LogisticRegression(
            max_iter=1000,
            C=1.0,
            random_state=RANDOM_STATE,
            solver='lbfgs'
        ))
    ])

    return pipeline


def generar_datos_sinteticos_m1():
    """Genera dataset multilingüe (es/pt/en) para validar M1."""
    datos = [
        ("COMPRA EN IFOOD RIO DE JANEIRO", "alimentacion"),
        ("UBER TRIP SAN PAULO", "transporte"),
        ("PAGO DE CONTA DE LUZ LIGHT ENEL", "servicios"),
        ("DEPOSITO NOMINA EMPRESA QUINCENA", "ingresos"),
        ("TRANSFERENCIA AHORRO BANAMEX", "ahorro_inversion"),
        ("CONSULTA MEDICA FARMACIA SIMILARES", "salud"),
        ("PAGO DE COLEGIATURA Y LIBROS", "educacion"),
        ("BOLETOS DE CINEPOLIS Y PALOMITAS", "entretenimiento"),
        ("COMPRA DE ROPA EN ZARA", "compras"),
        ("PAGO COMISION BANCO TPV", "finanzas"),
        ("RENTA DE DEPARTAMENTO MENSUAL", "vivienda"),
        ("GASTO DIVERSO SIN ESPECIFICAR", "otros"),
        ("IFOOD BRASIL RESTAURANTE", "alimentacion"),
        ("SUPERMERCADO WALMART COMPRAS", "alimentacion"),
        ("GASOLINERA PEMEX CARGA", "transporte"),
        ("PAGO SERVICIO INTERNET TELMEX", "servicios"),
    ]
    df = pd.DataFrame(datos, columns=["Descripcion_Transaccion", "categoria_slug"])
    return df


def entrenar_y_evaluar_m1(df):
    print("--- ENTRENANDO MODELO M1 (CLASIFICADOR DE CATEGORÍAS) ---")
    X = df["Descripcion_Transaccion"]
    y = df["categoria_slug"]

    pipeline_m1 = construir_pipeline_m1()
    pipeline_m1.fit(X, y)

    preds = pipeline_m1.predict(X)
    print("Accuracy M1:", accuracy_score(y, preds))
    print("\nReporte de Clasificación M1:")
    print(classification_report(y, preds, zero_division=0))

    return pipeline_m1


# ==============================================================================
# 3. MODELO M2: CLASIFICADOR DE PERFIL DE SALUD FINANCIERA (8 RATIOS)
# ==============================================================================

def construir_pipeline_m2():
    """
    Construye el modelo M2 entrenado EXCLUSIVAMENTE sobre los 8 ratios relativos:
    1. ratio_ahorro
    2. ratio_vivienda
    3. ratio_deuda
    4. ratio_gasto_esencial
    5. ratio_gasto_discrecional
    6. ratio_fondo_emergencia
    7. ratio_cobertura_ingresos
    8. ratio_margen_neto
    """
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=RANDOM_STATE
    )
    return model


def generar_datos_sinteticos_m2(n_samples=300):
    """Genera datos de entrenamiento usando únicamente los 8 ratios relativos."""
    np.random.seed(RANDOM_STATE)

    r_ahorro = np.random.uniform(0.0, 0.4, n_samples)
    r_vivienda = np.random.uniform(0.1, 0.5, n_samples)
    r_deuda = np.random.uniform(0.0, 0.6, n_samples)
    r_esencial = np.random.uniform(0.2, 0.7, n_samples)
    r_discrecional = np.random.uniform(0.05, 0.4, n_samples)
    r_fondo_emergencia = np.random.uniform(0.0, 6.0, n_samples) # meses
    r_cobertura = np.random.uniform(0.5, 2.0, n_samples)
    r_margen_neto = np.random.uniform(-0.2, 0.5, n_samples)

    X = np.column_stack([
        r_ahorro, r_vivienda, r_deuda, r_esencial,
        r_discrecional, r_fondo_emergencia, r_cobertura, r_margen_neto
    ])

    # Lógica sintética para etiquetar los perfiles
    y = []
    for i in range(n_samples):
        if r_ahorro[i] >= 0.15 and r_deuda[i] <= 0.30 and r_fondo_emergencia[i] >= 3.0:
            y.append("saludable")
        elif r_deuda[i] > 0.45 or r_margen_neto[i] < 0.0 or r_fondo_emergencia[i] < 1.0:
            y.append("en_riesgo")
        else:
            y.append("en_observacion")

    columns = [
        "ratio_ahorro", "ratio_vivienda", "ratio_deuda", "ratio_gasto_esencial",
        "ratio_gasto_discrecional", "ratio_fondo_emergencia",
        "ratio_cobertura_ingresos", "ratio_margen_neto"
    ]

    return pd.DataFrame(X, columns=columns), np.array(y)


def entrenar_y_evaluar_m2(X, y):
    print("\n--- ENTRENANDO MODELO M2 (PERFIL DE SALUD FINANCIERA) ---")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    pipeline_m2 = construir_pipeline_m2()
    pipeline_m2.fit(X_train, y_train)

    preds = pipeline_m2.predict(X_test)
    print("Accuracy M2:", accuracy_score(y_test, preds))
    print("\nReporte de Clasificación M2:")
    print(classification_report(y_test, preds, zero_division=0))

    return pipeline_m2


# ==============================================================================
# 4. EXPORTACIÓN DE ARCHIVOS .PKL Y VERIFICACIÓN
# ==============================================================================

def exportar_y_probar_modelos(modelo_m1, modelo_m2):
    print("\n--- EXPORTANDO Y PROBANDO ARCHIVOS .PKL ---")

    filename_m1 = "modelo_clasificador_salud_financiera.pkl"
    filename_m2 = "modelo_perfil_salud.pkl"

    # 1. Guardar archivos pickle
    with open(filename_m1, "wb") as f:
        pickle.dump(modelo_m1, f)

    with open(filename_m2, "wb") as f:
        pickle.dump(modelo_m2, f)

    print(f"[OK] Archivo generado: {filename_m1}")
    print(f"[OK] Archivo generado: {filename_m2}")

    # 2. Cargar y simular llamada desde FastAPI
    with open(filename_m1, "rb") as f:
        m1_loaded = pickle.load(f)

    with open(filename_m2, "rb") as f:
        m2_loaded = pickle.load(f)

    # --- Prueba M1 ---
    test_desc = ["COMPRA EN IFOOD BRASIL RIO", "PAGO SERVICIO ENEL LUZ"]
    m1_preds = m1_loaded.predict(test_desc)
    m1_probs = m1_loaded.predict_proba(test_desc)
    m1_confianza = [float(np.max(p)) for p in m1_probs]

    print("\nPrueba de inferencia M1 (FastAPI/Java):")
    for desc, pred, conf in zip(test_desc, m1_preds, m1_confianza):
        print(f"  Texto: '{desc}' -> Predicción: '{pred}' (Confianza: {conf:.2%})")

    # --- Prueba M2 ---
    # Array de 8 ratios: [ahorro, vivienda, deuda, esencial, discrecional, fondo, cobertura, margen]
    test_ratios = np.array([
        [0.20, 0.25, 0.15, 0.40, 0.15, 4.0, 1.3, 0.20],  # Perfil saludable
        [0.02, 0.45, 0.50, 0.70, 0.20, 0.5, 0.8, -0.10]  # Perfil en riesgo
    ])

    m2_preds = m2_loaded.predict(test_ratios)
    m2_probs = m2_loaded.predict_proba(test_ratios)

    print("\nPrueba de inferencia M2 (FastAPI/Java):")
    for ratios, pred, proba in zip(test_ratios, m2_preds, m2_probs):
        confianza = float(np.max(proba))
        print(f"  Ratios: {ratios.tolist()} -> Perfil: '{pred}' (Confianza: {confianza:.2%})")


if __name__ == "__main__":
    # Ejecutar pipeline M1
    df_m1 = generar_datos_sinteticos_m1()
    modelo_m1 = entrenar_y_evaluar_m1(df_m1)

    # Ejecutar pipeline M2
    X_m2, y_m2 = generar_datos_sinteticos_m2()
    modelo_m2 = entrenar_y_evaluar_m2(X_m2, y_m2)

    # Guardar y probar
    exportar_y_probar_modelos(modelo_m1, modelo_m2)

--- ENTRENANDO MODELO M1 (CLASIFICADOR DE CATEGORÍAS) ---
Accuracy M1: 1.0

Reporte de Clasificación M1:
                  precision    recall  f1-score   support

ahorro_inversion       1.00      1.00      1.00         1
    alimentacion       1.00      1.00      1.00         3
         compras       1.00      1.00      1.00         1
       educacion       1.00      1.00      1.00         1
 entretenimiento       1.00      1.00      1.00         1
        finanzas       1.00      1.00      1.00         1
        ingresos       1.00      1.00      1.00         1
           otros       1.00      1.00      1.00         1
           salud       1.00      1.00      1.00         1
       servicios       1.00      1.00      1.00         2
      transporte       1.00      1.00      1.00         2
        vivienda       1.00      1.00      1.00         1

        accuracy                           1.00        16
       macro avg       1.00      1.00      1.00        16
    weighted avg       

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
